# Day 18 — Practice Session · **SOLUTIONS**
### Merge, Join & Concat · Python for Data Science

**Prepared by Srinivasa Sai Chava**  ·  Boston University

---

> **Instructor copy.** Every question is followed by the answer and the reasoning.
> The student copy (`Day18_Practice_Questions.ipynb`) is identical minus the answer blocks.

| Part | Focus | Questions |
|---|---|---|
| A | Predict the output | 8 |
| B | Spot & fix the bug | 4 |
| C | Write the code | 5 |
| D | Challenge | 2 |

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5],
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran"],
    "dept_id":[10, 20, 10, 30, 20],
})

salaries = pd.DataFrame({
    "emp_id": [1, 2, 3, 6],
    "salary": [52000, 61000, 48000, 70000],
})

depts = pd.DataFrame({
    "dept_id": [10, 20, 40],
    "dept":    ["Sales", "IT", "Legal"],
})

print("employees:", len(employees), "| salaries:", len(salaries), "| depts:", len(depts))
employees

---
## Setup — run this first

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.simplefilter("ignore")

employees = pd.DataFrame({
    "emp_id": [1, 2, 3, 4, 5],
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran"],
    "dept_id":[10, 20, 10, 30, 20],
})

salaries = pd.DataFrame({
    "emp_id": [1, 2, 3, 6],
    "salary": [52000, 61000, 48000, 70000],
})

depts = pd.DataFrame({
    "dept_id": [10, 20, 40],
    "dept":    ["Sales", "IT", "Legal"],
})

print("employees:", len(employees), "| salaries:", len(salaries), "| depts:", len(depts))
employees

---
# Part A — Predict the Output  *(8 min)*

---
# Part A — Predict the Output

**Teaching note:** A1–A3 establish the row-count instinct. A7 (row explosion) is the one
that matters most in real work.

### A1. How many rows?

In [ ]:
m = employees.merge(salaries, on="emp_id")
print(len(m))
print(m["name"].tolist())

*Your prediction:*  

> ### ✅ Answer A1
> ```
> 3
> ['Ravi', 'Sara', 'Amit']
> ```
> **Why:** `merge` defaults to `how="inner"`, so only `emp_id` values present in **both**
> tables survive. Neha (4) and Kiran (5) have no salary; employee 6 has a salary but no
> record.
>
> **Five employees went in, three came out — and nothing warned you.** That is why the
> default being `inner` is worth remembering: it is a silent filter.

### A2. Now how many?

In [ ]:
m = employees.merge(salaries, on="emp_id", how="left")
print(len(m))
print(m["salary"].isna().sum())

*Your prediction:*  

> ### ✅ Answer A2
> ```
> 5
> 2
> ```
> **Why:** `how="left"` keeps **every** row of the left table. Neha and Kiran survive with
> `NaN` in the `salary` column.
>
> This is usually what you want when the left table is your population — you asked about
> employees, so every employee should appear in the answer, even those whose salary is
> unknown. The two `NaN`s are information, not a failure.

### A3. And these two?

In [ ]:
print(len(employees.merge(salaries, on="emp_id", how="right")))
print(len(employees.merge(salaries, on="emp_id", how="outer")))

*Your prediction:*  

> ### ✅ Answer A3
> ```
> 4
> 6
> ```
> **Why:** `right` keeps all four salary rows (ids 1, 2, 3, 6), with `NaN` names for id 6.
> `outer` keeps every id from either side — 1, 2, 3, 4, 5, 6 — which is six rows.
>
> Four joins, four different row counts from the same two tables: **3, 5, 4, 6**. The join
> type is not a formality; it changes the answer.

### A4. What is the dtype now?

In [ ]:
print(salaries["salary"].dtype)
m = employees.merge(salaries, on="emp_id", how="left")
print(m["salary"].dtype)

*Your prediction:*  

> ### ✅ Answer A4
> ```
> int64
> float64
> ```
> **Why:** the left join introduced two `NaN`s, and **an integer column cannot hold `NaN`**
> — so the whole column was converted to float. Salaries now print as `52000.0`.
>
> Day 14's rule, arriving through a new door. If an ID or a count suddenly grows a decimal
> point, an unmatched merge is a likely cause.

### A5. What does the indicator show?

In [ ]:
chk = employees.merge(salaries, on="emp_id", how="outer", indicator=True)
print(chk["_merge"].value_counts())

*Your prediction:*  

> ### ✅ Answer A5
> ```
> _merge
> both          3
> left_only     2
> right_only    1
> Name: count, dtype: int64
> ```
> **Why:** three ids matched, two employees had no salary, one salary had no employee.
>
> **This is the diagnostic to run first**, before choosing a real join. It answers "how much
> will I lose, and from which side?" in one line — and the answer often changes which join
> you want. A `right_only` row is usually a data problem worth investigating, not something
> to quietly drop.

### A6. What are the column names?

In [ ]:
a = pd.DataFrame({"id": [1, 2], "value": [10, 20]})
b = pd.DataFrame({"id": [1, 2], "value": [99, 98]})
print(a.merge(b, on="id").columns.tolist())

*Your prediction:*  

> ### ✅ Answer A6
> ```
> ['id', 'value_x', 'value_y']
> ```
> **Why:** both tables have a `value` column, so pandas keeps both and tags them `_x` (left)
> and `_y` (right).
>
> It works, but six months later nobody remembers which was which. Always pass meaningful
> suffixes: `suffixes=("_before", "_after")`.

### A7. Count the rows carefully.

In [ ]:
orders = pd.DataFrame({"cust": [1, 1, 2], "amt": [10, 20, 30]})
lookup = pd.DataFrame({"cust": [1, 1, 2], "name": ["A", "A", "B"]})
m = orders.merge(lookup, on="cust")
print(len(m))
print(m["amt"].sum())

*Your prediction:*  

> ### ✅ Answer A7
> ```
> 5
> 90
> ```
> **Three rows merged with three rows gives five.** Customer 1 appears **twice on each
> side**, so 2 × 2 = 4 rows come out of that key alone, plus one for customer 2.
>
> **And the total is now wrong.** The true sum of amounts is 10 + 20 + 30 = **60**, but the
> merged table reports **90** — because customer 1's amounts of 10 and 20 each appear twice.
> That inflated 90 is what would reach a report, a dashboard or an invoice.
>
> **No error. No warning.** A duplicate in a lookup table is easy to create and invisible
> once merged. `validate="many_to_one"` would have raised a `MergeError` instead.

### A8. What does concat give?

In [ ]:
a = pd.DataFrame({"name": ["Ravi"], "mark": [88]})
b = pd.DataFrame({"name": ["Sara"], "mark": [91]})
print(pd.concat([a, b]).index.tolist())
print(pd.concat([a, b], ignore_index=True).index.tolist())

*Your prediction:*  

> ### ✅ Answer A8
> ```
> [0, 0]
> [0, 1]
> ```
> **Why:** `concat` keeps each table's original index, so stacking two one-row tables gives
> **two rows both labelled 0**.
>
> Every Day 16 trap returns at once: `loc[0]` now matches two rows, and labels no longer
> correspond to positions. **Pass `ignore_index=True`** whenever the old labels mean nothing
> — which, when stacking files, they never do.

---
# Part B — Spot & Fix the Bug  *(7 min)*

---
# Part B — Spot & Fix the Bug

**Teaching note:** B1 and B3 both produce wrong numbers with no error. Those are the ones
worth the time.

### B1. This should report the average salary per department.

In [ ]:
df = employees.merge(salaries, on="emp_id").merge(depts, on="dept_id")
print("rows:", len(df))
print(df.groupby("dept")["salary"].mean())

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B1
> **Symptom:** the report covers **3 of 5 employees**, and nothing says so.
> **Cause:** `merge` defaults to `inner`. The salary merge silently drops Neha and Kiran,
> who have no salary row — 5 rows become 3.
>
> The department merge happens to lose nobody here, because the three survivors are all in
> departments 10 and 20. But that is luck: Neha's department 30 is **not** in `depts` at
> all, so had she survived the first merge she would have been dropped by the second.
> A report built this way is one data change away from being wrong again.

In [ ]:
# Show the damage at each step
step1 = employees.merge(salaries, on="emp_id")
step2 = step1.merge(depts, on="dept_id")
print(f"employees {len(employees)} -> after salaries {len(step1)} -> after depts {len(step2)}")
# 5 -> 3 -> 3   the salary merge did the damage; the dept merge got lucky

# Diagnose properly
print()
print(employees.merge(salaries, on="emp_id", how="outer", indicator=True)["_merge"].value_counts())
print()
print(employees.merge(depts, on="dept_id", how="left", indicator=True)["_merge"].value_counts())

# THE FIX: left joins, so every employee survives, then report the gaps
before = len(employees)
df = (employees
      .merge(salaries, on="emp_id", how="left", validate="one_to_one")
      .merge(depts,    on="dept_id", how="left", validate="many_to_one"))
assert len(df) == before

print()
print("employees with no salary   :", df["salary"].isna().sum())
print("employees with no dept name:", df["dept"].isna().sum())
print()
print(df.groupby("dept")["salary"].mean())
print("...computed from", df["salary"].notna().sum(), "of", before, "employees")

### B2. This should attach the department name.

In [ ]:
df = employees.merge(depts, on="dept")
print(df)

*Error:* 

*Your fix:*

> ### ✅ Answer B2
> **Error:** `KeyError: 'dept'`
> **Cause:** `employees` has no `dept` column — it has `dept_id`. The shared key is
> `dept_id`; `dept` is what you are trying to *fetch*.

In [ ]:
print("employees columns:", employees.columns.tolist())
print("depts columns    :", depts.columns.tolist())
print("shared           :", set(employees.columns) & set(depts.columns))
print()

df = employees.merge(depts, on="dept_id", how="left")
print(df)

# When a merge raises KeyError, print both column lists. The set
# intersection shows you immediately what the two tables actually share.

### B3. This should pair each employee with their salary.

In [ ]:
df = pd.concat([employees, salaries], axis=1)
print(df[["name", "emp_id", "salary"]].head())

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B3
> **Symptom:** the salaries are attached to the **wrong people**, and there is no error.
> **Cause:** `concat(axis=1)` glues columns side by side by **index position**, not by key.
> Row 3 of `employees` (Neha) gets row 3 of `salaries` (employee **6**'s 70000) purely
> because they sit at the same position.

In [ ]:
wrong = pd.concat([employees, salaries], axis=1)
print("what concat produced:")
print(wrong)
print()
print("Neha's emp_id is 4, but she has been given employee 6's salary.")
print("Note there are now TWO emp_id columns that disagree.")
print()

# THE FIX: merge on the key
right = employees.merge(salaries, on="emp_id", how="left")
print("what merge produces:")
print(right)

# THE RULE:
#   same things, different information  -> merge, on a key
#   different things, same information  -> concat, stacking rows
#
# concat(axis=1) is only safe when you already know the rows line up.

### B4. This should stack two months of data.

In [ ]:
jan = pd.DataFrame({"name": ["Ravi", "Sara"], "sales": [100, 200]})
feb = pd.DataFrame({"name": ["Ravi", "Amit"], "sales": [150, 120]})

both = pd.concat([jan, feb])
print(both)
print(both.loc[0])

*What's wrong:* 

*Your fix:*

> ### ✅ Answer B4
> **Symptom:** `both.loc[0]` returns **two rows**, not one — the index has duplicate labels
> 0, 1, 0, 1.
> **Cause:** `concat` preserved each table's original index.

In [ ]:
jan = pd.DataFrame({"name": ["Ravi", "Sara"], "sales": [100, 200]})
feb = pd.DataFrame({"name": ["Ravi", "Amit"], "sales": [150, 120]})

both = pd.concat([jan, feb], ignore_index=True)
print(both)
print()
print("index:", both.index.tolist())
print("loc[0] is now one row:", both.loc[0, "name"])

# Note Ravi correctly appears TWICE - two months of his sales. That is not
# a duplicate to remove, it is the point of stacking. If you want his total:
print()
print(both.groupby("name")["sales"].sum())

---
# Part C — Write the Code  *(12 min)*

---
# Part C — Write the Code

**Teaching note:** C3 is the audit habit. C5 is the pattern they will reuse for the rest of
the course.

### C1. Keep everyone
Join `employees` to `salaries` so that **every employee** appears, and print how many have no
salary recorded.

In [ ]:
# your code here

In [ ]:
df = employees.merge(salaries, on="emp_id", how="left")

print("rows:", len(df), "(employees:", len(employees), ")")
print("no salary recorded:", df["salary"].isna().sum())
print()
print(df)

# rows: 5 (employees: 5)
# no salary recorded: 2
#
# how="left" because employees is the population we were asked about.
# Neha and Kiran stay in the answer with NaN salaries.

### C2. Find the mismatches
Using one merge, list which `emp_id` values appear in only one of the two tables, and say
which side each came from.

In [ ]:
# your code here

In [ ]:
chk = employees.merge(salaries, on="emp_id", how="outer", indicator=True)

print(chk["_merge"].value_counts())
print()
print(chk[chk["_merge"] != "both"][["emp_id", "name", "salary", "_merge"]])

#    emp_id   name   salary    _merge
# 3       4   Neha      NaN left_only
# 4       5  Kiran      NaN left_only
# 5       6    NaN  70000.0 right_only
#
# Employees 4 and 5 have no salary row.
# Salary row 6 has no matching employee - which is a DATA PROBLEM worth
# raising, not something to quietly drop with an inner join.

### C3. A merge that cannot lie
Write a function `safe_merge(left, right, **kwargs)` that merges and raises if the row count
changes. Test it on a merge that is fine, and one that is not.

In [ ]:
orders = pd.DataFrame({"cust": [1, 1, 2], "amt": [10, 20, 30]})
good   = pd.DataFrame({"cust": [1, 2],    "name": ["A", "B"]})
bad    = pd.DataFrame({"cust": [1, 1, 2], "name": ["A", "A-dup", "B"]})

# your code here

In [ ]:
orders = pd.DataFrame({"cust": [1, 1, 2], "amt": [10, 20, 30]})
good   = pd.DataFrame({"cust": [1, 2],    "name": ["A", "B"]})
bad    = pd.DataFrame({"cust": [1, 1, 2], "name": ["A", "A-dup", "B"]})


def safe_merge(left, right, **kwargs):
    """Merge, refusing to silently change the row count."""
    out = left.merge(right, **kwargs)
    if len(out) != len(left):
        raise ValueError(f"row count changed: {len(left)} -> {len(out)}")
    return out


print("clean lookup:")
print(safe_merge(orders, good, on="cust", how="left"))

print()
try:
    safe_merge(orders, bad, on="cust", how="left")
except ValueError as e:
    print("ValueError:", e)

# row count changed: 3 -> 5
#
# Day 11's custom exceptions doing real work. An error that fires is
# infinitely better than a report that is quietly inflated.

### C4. Stack and summarise
Stack the three monthly tables below into one, renumber the index, and print total sales per
person.

In [ ]:
jan = pd.DataFrame({"name": ["Ravi", "Sara"], "sales": [100, 200]})
feb = pd.DataFrame({"name": ["Ravi", "Amit"], "sales": [150, 120]})
mar = pd.DataFrame({"name": ["Sara", "Amit"], "sales": [180, 90]})

# your code here

In [ ]:
jan = pd.DataFrame({"name": ["Ravi", "Sara"], "sales": [100, 200]})
feb = pd.DataFrame({"name": ["Ravi", "Amit"], "sales": [150, 120]})
mar = pd.DataFrame({"name": ["Sara", "Amit"], "sales": [180, 90]})

all_months = pd.concat([jan, feb, mar], ignore_index=True)
print(all_months)
print("rows:", len(all_months), "=", len(jan) + len(feb) + len(mar))
print()
print(all_months.groupby("name")["sales"].sum().sort_values(ascending=False))

# Amit    210
# Ravi    250
# Sara    380
#
# concat, not merge - three tables describing DIFFERENT rows with the SAME
# columns. Adding a month column first would make the result more useful:
tagged = pd.concat([jan.assign(month="Jan"), feb.assign(month="Feb"),
                    mar.assign(month="Mar")], ignore_index=True)
print()
print(tagged.pivot_table(index="name", columns="month", values="sales", fill_value=0))

### C5. Three tables, one report
Combine `employees`, `salaries` and `depts` so every employee appears, then print the average
salary per department **and** how many employees are unaccounted for at each step.

In [ ]:
# your code here

In [ ]:
before = len(employees)

df = (employees
      .merge(salaries, on="emp_id",  how="left", validate="one_to_one")
      .merge(depts,    on="dept_id", how="left", validate="many_to_one"))

assert len(df) == before, "the merges changed the row count"

print("employees            :", before)
print("without a salary     :", df["salary"].isna().sum())
print("without a dept name  :", df["dept"].isna().sum())
print()
print(df)
print()
print(df.groupby("dept")["salary"].mean())
print("computed from", df["salary"].notna().sum(), "of", before, "employees")

# employees            : 5
# without a salary     : 2
# without a dept name  : 1      <- Neha's dept_id 30 is not in depts
#
# dept
# IT       61000.0
# Sales    50000.0
#
# The two validate values differ for a reason:
#   one_to_one   each employee appears once in each table
#   many_to_one  many employees share a dept, each dept listed once

---
# Part D — Challenge  *(3 min, or take home)*

---
# Part D — Challenge

**Teaching note:** D1 is the detective exercise — finding the duplicate before it does
damage. D2 shows the four joins as four different business answers.

### D1. Find the culprit
The merge below produces more rows than it should. **Without** looking at the tables by eye,
write code that finds which key values are duplicated and in which table.

In [ ]:
left  = pd.DataFrame({"k": [1, 2, 3, 4],       "a": [10, 20, 30, 40]})
right = pd.DataFrame({"k": [1, 2, 2, 3, 3, 3], "b": ["x", "y", "y2", "z", "z2", "z3"]})

# your code here

In [ ]:
left  = pd.DataFrame({"k": [1, 2, 3, 4],       "a": [10, 20, 30, 40]})
right = pd.DataFrame({"k": [1, 2, 2, 3, 3, 3], "b": ["x", "y", "y2", "z", "z2", "z3"]})

merged = left.merge(right, on="k")
print(f"left {len(left)} rows -> merged {len(merged)} rows")
print()

# 1. Is the key unique in each table?
print("k unique in left :", left["k"].is_unique)
print("k unique in right:", right["k"].is_unique)
print()

# 2. WHICH values repeat, and how often?
dups = right["k"].value_counts()
print("repeated keys in right:")
print(dups[dups > 1])
print()

# 3. Predict the merged size before running it
expected = left["k"].map(right["k"].value_counts()).fillna(0).sum()
print("predicted merged rows:", int(expected), " actual:", len(merged))

# k unique in left : True
# k unique in right: False
# repeated keys in right:  k=3 three times, k=2 twice
# predicted 6, actual 6
#
# key 1 -> 1 row, key 2 -> 2 rows, key 3 -> 3 rows, key 4 -> 0 rows = 6
#
# is_unique on the key of each table is a two-second check that would have
# prevented this. Run it before any merge you are unsure about.

### D2. Four joins, four business answers
For `employees` and `salaries`, run all four joins. For each, print the row count and write
one sentence saying **what business question that join answers**.

In [ ]:
# your code here

In [ ]:
questions = {
    "inner": "Employees we can report a salary for - complete records only.",
    "left":  "All our employees, with salary where we have it. The usual choice.",
    "right": "All salary records, flagging any with no matching employee.",
    "outer": "Everything from both, for auditing what is missing on each side.",
}

for how in ["inner", "left", "right", "outer"]:
    m = employees.merge(salaries, on="emp_id", how=how)
    print(f"{how:6}  {len(m)} rows   {questions[how]}")

# inner   3 rows   Employees we can report a salary for - complete records only.
# left    5 rows   All our employees, with salary where we have it. The usual choice.
# right   4 rows   All salary records, flagging any with no matching employee.
# outer   6 rows   Everything from both, for auditing what is missing on each side.
#
# THE POINT: 3, 5, 4 and 6 are all correct answers - to four DIFFERENT
# questions. Choosing a join is choosing which question you are answering,
# and inner being the default means that choice is easy to make by accident.
print()
print("Of these, the one that reports a company-wide average over only 3 of 5")
print("employees, without saying so, is the one that ends up in a bad report.")

---
## Done? Self-check

- [ ] I know `merge` defaults to `inner` and what that silently does
- [ ] I can say what each of the four joins keeps
- [ ] I ran `outer` + `indicator=True` at least once to diagnose
- [ ] I know why an unmatched merge turns an int column into float
- [ ] I know how duplicate keys multiply rows
- [ ] I checked `len()` before and after every merge
- [ ] I know when to use `concat` instead of `merge`

### Homework
1. Take two CSVs and run all four joins, recording the row count of each.
2. Deliberately duplicate a key and watch the row count explode.
3. Rebuild Day 9's city lookup as a merge instead of a dictionary.

### Next class — Topic 1.18: Data cleaning — missing values & duplicates
Finding them, deciding what to do, and the cost of each choice.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*

---
## Wrap-up — running the last 5 minutes

Three cold-call questions:

1. *"What is `merge`'s default `how`?"* → `inner` — a silent filter.
2. *"Three rows merged with three rows gives how many?"* → possibly more than three.
3. *"`concat(axis=1)` matches on what?"* → the index, not a key.

**Common misconceptions to watch for today**

| Misconception | Correction |
|---|---|
| "A merge cannot lose rows" | `inner` is the default and drops non-matches silently |
| "A merge cannot add rows" | Duplicate keys multiply them |
| "`NaN` after a merge is a bug" | It is the correct answer for an unmatched row |
| "`concat(axis=1)` joins on a key" | It aligns on the **index** |
| "`concat` keeps a clean index" | Not without `ignore_index=True` |
| "The join type is a technical detail" | It decides which question you are answering |

**D2 is the one to close on.** The same two tables give 3, 5, 4 and 6 rows — four correct
answers to four different questions. Students who see that stop treating `how=` as
boilerplate.

**A7 deserves a live run.** Three rows and three rows producing five, with no error, is the
most convincing argument for the row-count habit that exists. Then show the totals: the true
sum is **60**, the merged sum is **90**. A 50% overstatement, arrived at silently. Ask the
class where that number would have ended up — a report, an invoice, a forecast.

**The habit to send them away with:** `assert len(after) == len(before)` beside every merge.
It is one line, and it converts the two most damaging merge bugs — silent drops and silent
multiplication — into loud failures.

**Homework given:** all four joins with row counts; deliberately explode a merge; rebuild
Day 9's lookup as a merge.

**Next session:** Topic 1.18 — data cleaning. Today *created* `NaN`s through unmatched
joins; tomorrow is about deciding what to do with them, and what each choice costs.